# S8 - Actividad autónoma: streaming con Spark sobre datos reales de mi proyecto

**Fuente real:** ESP32 físico + sensor DHT11 → MQTT (`test.mosquitto.org`) → bridge → Kafka, topic **`clima-real-eventos`**.
**Notebook:** `08_streaming_estructurado_autonoma` · **Entorno:** lambda26 (`pyspark`, red `lambda26-kafka-net`).

**Antes de ejecutar (fuera del notebook), las tres cosas tienen que estar vivas:**
1. `kafka` y `uso-atmos` arriba (`docker compose -f kafka/compose.yml up -d` y `docker compose -f uso-atmos/compose.yml up -d`).
2. El **ESP32 publicando**: Monitor Serial del Arduino IDE mostrando `Publicado: {...}` cada 3 s.
3. El **bridge corriendo** en su propia terminal: `docker compose -f uso-atmos/compose.yml exec -e MQTT_TOPIC=lambda26/atmos-real/equipo01/lecturas -e KAFKA_TOPIC_ATMOS=clima-real-eventos uso-atmos python /app/bridge_mqtt_kafka.py`

**Cómo usarlo:** ejecuta las celdas en orden. Las tareas 2, 3 y 4 comparten consultas que siguen vivas: **captura la evidencia de cada una antes de pasar a la tarea 5**, que detiene todo.

> Los `Batch: N` del sink `console` no aparecen en Jupyter. Por eso este notebook usa el sink `memory` y consulta los resultados con `spark.sql(...)`: así la evidencia queda **visible en la celda** y se puede capturar. La interfaz de Spark está en http://localhost:4042.

## Parámetros

In [1]:
# ---- Parámetros de TU proyecto (cambia solo lo que hayas cambiado en el sketch o el bridge) ----
TOPIC = "clima-real-eventos"
BOOTSTRAP = "kafka:9092"
SENSOR_ID = "esp32-fisico-equipo01"   # = "esp32-fisico-" + el EQUIPO que pusiste en el sketch
VENTANA = "20 seconds"                # ancho de la ventana de agregación
MARGEN_WATERMARK = "10 seconds"       # cuánto se espera, como máximo, por un dato tardío
RUTA_PARQUET = "./artifacts/clima_parquet_agg"
CHK_PARQUET = "./artifacts/chk_clima_agg"

## SparkSession con el conector de Kafka

## Utilidades
Esquema del evento y funciones auxiliares (lectura batch del topic, publicación de eventos de prueba, detener consultas).

In [3]:
import os, time, json, shutil
from pyspark.sql import functions as F
from pyspark.sql.functions import col, from_json, window, avg, count, to_json, struct
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

# Contrato de TU evento (sin "presion": el DHT11 no la mide y no se inventa)
esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("temperatura", DoubleType())
    .add("humedad", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)


def detener_todas():
    for q in spark.streams.active:
        print("deteniendo consulta:", q.name or q.id)
        q.stop()


def reiniciar_checkpoint(*rutas):
    for r in rutas:
        shutil.rmtree(r, ignore_errors=True)


def _crudo_batch():
    return (
        spark.read.format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP)
        .option("subscribe", TOPIC)
        .option("startingOffsets", "earliest")
        .load()
    )


def leer_topico_batch():
    """Lee TODO el topic (lectura batch) ya parseado, con el timestamp de Kafka como kafka_ts."""
    return (
        _crudo_batch()
        .select(
            from_json(col("value").cast("string"), esquema_evento).alias("e"),
            col("offset"),
            col("timestamp").alias("kafka_ts"),
        )
        .select("e.*", "offset", "kafka_ts")
        .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
    )


def ultimo_evento_real():
    fila = (
        leer_topico_batch()
        .where(col("origen") == "esp32-fisico")
        .orderBy(col("offset").desc())
        .limit(1)
        .collect()
    )
    if not fila:
        raise RuntimeError("No hay lecturas reales en el topic: ¿el ESP32 está publicando y el bridge corriendo?")
    return fila[0]


def _armar_mensaje(sensor_id, temperatura, humedad, ts_ms, origen):
    columnas = ["tipoEvento", "sensorId", "temperatura", "humedad", "origen", "timestamp"]
    df = spark.createDataFrame(
        [("sensor.lectura", sensor_id, float(temperatura), float(humedad), origen, int(ts_ms))],
        columnas,
    )
    return df.select(
        col("sensorId").cast("string").alias("key"),
        to_json(struct(*columnas)).alias("value"),
    )


def publicar_evento(sensor_id, temperatura, humedad, ts_ms, origen):
    (
        _armar_mensaje(sensor_id, temperatura, humedad, ts_ms, origen)
        .write.format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP)
        .option("topic", TOPIC)
        .save()
    )

## Tarea 1 - Lectura streaming con esquema explícito, sobre datos reales
**Producto:** el stream de `clima-real-eventos` parseado con un esquema explícito (como 3.3-3.4 de la guía), y evidencia de que los datos vienen del sensor físico, no de un simulador.

In [15]:
# Evidencia de datos REALES: de dónde vienen los eventos que hay en el topic
topico = leer_topico_batch()
print("mensajes en el topic:", topico.count())
(
    topico.groupBy("origen", "sensorId")
    .agg(F.count("*").alias("n"), F.min("ts").alias("primera_lectura"), F.max("ts").alias("ultima_lectura"))
    .show(truncate=False)
)
topico.orderBy(col("offset").desc()).select("offset", "sensorId", "temperatura", "humedad", "origen", "ts").show(5, truncate=False)

mensajes en el topic: 998


+----------------+-------------------------------+---+-------------------+-------------------+
|origen          |sensorId                       |n  |primera_lectura    |ultima_lectura     |
+----------------+-------------------------------+---+-------------------+-------------------+
|prueba-control  |esp32-fisico-equipo01-control  |1  |2026-10-07 10:19:40|2026-10-07 10:19:40|
|prueba-duplicado|esp32-fisico-equipo01-duplicado|2  |2026-10-07 10:22:25|2026-10-07 10:22:25|
|esp32-fisico    |esp32-fisico-equipo01          |994|2026-10-07 09:48:28|2026-10-07 10:46:29|
|prueba-tardio   |esp32-fisico-equipo01-tardio   |1  |2026-10-07 10:17:40|2026-10-07 10:17:40|
+----------------+-------------------------------+---+-------------------+-------------------+

+------+---------------------+-----------+-------+------------+-------------------+
|offset|sensorId             |temperatura|humedad|origen      |ts                 |
+------+---------------------+-----------+-------+------------+--------

In [25]:
crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
)

eventos = (
    crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema_evento).alias("e"))
    .select("e.*")
    .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))   # tiempo de EVENTO
)
print("es un stream:", eventos.isStreaming)
eventos.printSchema()

es un stream: True
root
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)



**Captura sugerida:** la celda de datos reales (`origen = esp32-fisico`) y el esquema parseado con `es un stream: True`.

## Elegir el margen del watermark con datos, no a ojo
El watermark debe cubrir el **retraso real** entre que el ESP32 mide y el evento llega a Kafka (WiFi + broker público + bridge). Esta celda lo mide: `kafka_ts − ts` (el `ts` del ESP32 se trunca al segundo, así que el retraso puede estar sobreestimado hasta 1 s).

In [6]:
lat = (
    leer_topico_batch()
    .where(col("origen") == "esp32-fisico")
    .withColumn("latencia_s", col("kafka_ts").cast("double") - col("ts").cast("double"))
)
lat.selectExpr(
    "count(*) AS n",
    "round(min(latencia_s), 2) AS min_s",
    "round(percentile_approx(latencia_s, 0.5), 2) AS p50_s",
    "round(percentile_approx(latencia_s, 0.95), 2) AS p95_s",
    "round(max(latencia_s), 2) AS max_s",
).show()

[Stage 7:>                                                          (0 + 1) / 1]

+---+-----+-----+-----+-----+
|  n|min_s|p50_s|p95_s|max_s|
+---+-----+-----+-----+-----+
|575| 0.96|  1.0| 1.12| 1.51|
+---+-----+-----+-----+-----+



**Cómo usar el resultado:** el margen (`MARGEN_WATERMARK`, ahora 10 s) debería quedar **por encima del p95** y cerca de unas 3 veces el intervalo de publicación (3 s), para tolerar un reintento de WiFi o un retraso del broker. Anota el p95 medido: es el argumento numérico de tu justificación.

- **Qué se gana con un margen mayor:** menos datos tardíos perdidos.
- **Qué se pierde:** cada ventana tarda más en cerrarse (resultado final = fin de ventana + margen) y se guarda más estado en memoria.

## Tarea 2 - Agregación por ventana con watermarking
**Producto:** promedio, mínimo y máximo de temperatura (y humedad media) por ventana fija de 20 s, con watermark de 10 s. Modo `append`: una ventana se emite **una sola vez, cuando se cierra** (fin de ventana + margen), y no vuelve a aparecer.

**Por qué esta métrica:** el DHT11 cambia lento y mide con baja resolución; promediar por ventana de 20 s (≈ 6 lecturas) suaviza ese ruido y da una serie útil para monitoreo. Deja corriendo ~2 minutos para ver varias ventanas cerradas.

In [7]:
def construir_agregado(ev):
    return (
        ev.withWatermark("ts", MARGEN_WATERMARK)
        .groupBy(window(col("ts"), VENTANA), col("sensorId"))
        .agg(
            avg("temperatura").alias("temp_prom"),
            F.min("temperatura").alias("temp_min"),
            F.max("temperatura").alias("temp_max"),
            avg("humedad").alias("hum_prom"),
            count("*").alias("n"),
        )
    )


detener_todas()   # arranque limpio
consulta_agg = (
    construir_agregado(eventos).writeStream
    .outputMode("append")
    .format("memory").queryName("agg_clima")
    .trigger(processingTime="5 seconds")
    .start()
)
print("consulta_agg activa:", consulta_agg.isActive)

consulta_agg activa: True


In [16]:
# Ejecuta esta celda varias veces, con ~30 s entre una y otra: aparecen ventanas nuevas y ninguna se repite
spark.sql("""
    SELECT window.start AS inicio, window.end AS fin, sensorId, n,
           round(temp_prom, 2) AS temp_prom, temp_min, temp_max, round(hum_prom, 1) AS hum_prom
    FROM agg_clima ORDER BY inicio, sensorId
""").show(30, truncate=False)

p = consulta_agg.lastProgress
if p:
    print("batchId:", p["batchId"], "| filas de entrada del lote:", p["numInputRows"])
    print("eventTime (incluye el watermark):", p.get("eventTime"))

+-------------------+-------------------+-------------------------------+---+---------+--------+--------+--------+
|inicio             |fin                |sensorId                       |n  |temp_prom|temp_min|temp_max|hum_prom|
+-------------------+-------------------+-------------------------------+---+---------+--------+--------+--------+
|2026-10-07 10:19:20|2026-10-07 10:19:40|esp32-fisico-equipo01          |5  |15.7     |15.7    |15.7    |36.0    |
|2026-10-07 10:19:40|2026-10-07 10:20:00|esp32-fisico-equipo01          |7  |15.7     |15.7    |15.7    |36.0    |
|2026-10-07 10:19:40|2026-10-07 10:20:00|esp32-fisico-equipo01-control  |1  |15.7     |15.7    |15.7    |36.0    |
|2026-10-07 10:20:00|2026-10-07 10:20:20|esp32-fisico-equipo01          |7  |15.7     |15.7    |15.7    |36.0    |
|2026-10-07 10:20:20|2026-10-07 10:20:40|esp32-fisico-equipo01          |6  |15.7     |15.7    |15.7    |36.0    |
|2026-10-07 10:20:40|2026-10-07 10:21:00|esp32-fisico-equipo01          |7  |15.

In [19]:
from pyspark.sql.window import Window

w = Window.orderBy("offset")
(
    leer_topico_batch().where(col("origen") == "esp32-fisico")
    .withColumn("ts_anterior", F.lag("ts").over(w))
    .withColumn("salto_s", col("ts").cast("long") - col("ts_anterior").cast("long"))
    .where(col("salto_s") > 10)
    .select("offset", "ts_anterior", "ts", "salto_s", "kafka_ts")
    .show(truncate=False)
)

+------+-------------------+-------------------+-------+-----------------------+
|offset|ts_anterior        |ts                 |salto_s|kafka_ts               |
+------+-------------------+-------------------+-------+-----------------------+
|710   |2026-10-07 10:23:46|2026-10-07 10:24:01|15     |2026-10-07 10:24:01.979|
|755   |2026-10-07 10:26:13|2026-10-07 10:34:23|490    |2026-10-07 10:34:24.073|
+------+-------------------+-------------------+-------+-----------------------+



**Captura sugerida:** la tabla con 3 o más ventanas cerradas (cada una una sola vez) y la línea `eventTime` con el `watermark`.

## Tarea 3 - Un dato tardío que el watermark descarta
**Producto:** un evento con **valores reales** (la última lectura del sensor) pero con marca de tiempo 120 s en el pasado, publicado cuando el watermark ya avanzó. Se publica también un evento **de control** con la hora actual: si el de control aparece y el tardío no, la ausencia se debe al watermark y no a un error.

> **Arranque en frío:** esto solo funciona si `consulta_agg` (Tarea 2) lleva ya un par de minutos corriendo con datos reales; si el tardío fuera lo primero que ve la consulta, no se descartaría.
> Las marcas de tiempo se toman del **reloj del ESP32**, no del contenedor, para no mezclar dos relojes distintos.

In [9]:
assert consulta_agg.isActive, "Ejecuta primero la Tarea 2: consulta_agg debe estar corriendo"

real = ultimo_evento_real()
ts_ref = real["timestamp"]
print("Última lectura real:", real["sensorId"], "|", real["temperatura"], "°C |", real["humedad"], "%")

publicar_evento(SENSOR_ID + "-control", real["temperatura"], real["humedad"], ts_ref, "prueba-control")
publicar_evento(SENSOR_ID + "-tardio", real["temperatura"], real["humedad"], ts_ref - 120_000, "prueba-tardio")
print(">>> publicados: '-control' (hora actual) y '-tardio' (120 s en el pasado)")

Última lectura real: esp32-fisico-equipo01 | 15.7 °C | 36.0 %


[Stage 19:>                                                         (0 + 4) / 4]

>>> publicados: '-control' (hora actual) y '-tardio' (120 s en el pasado)


In [17]:
# Espera a que el watermark siga avanzando con las lecturas reales y cierre la ventana del control
time.sleep(60)

print("== Salida agregada: debe aparecer el CONTROL y NO el TARDÍO ==")
spark.sql("""
    SELECT sensorId, window.start AS inicio, window.end AS fin, n, round(temp_prom, 1) AS temp_prom
    FROM agg_clima WHERE sensorId LIKE '%-control' OR sensorId LIKE '%-tardio' ORDER BY sensorId
""").show(truncate=False)

print("== Pero SÍ se publicaron los dos en el topic ==")
(
    leer_topico_batch().where(col("origen").isin("prueba-control", "prueba-tardio"))
    .select("offset", "sensorId", "origen", "temperatura", "humedad", "ts").orderBy("offset").show(truncate=False)
)

== Salida agregada: debe aparecer el CONTROL y NO el TARDÍO ==
+-----------------------------+-------------------+-------------------+---+---------+
|sensorId                     |inicio             |fin                |n  |temp_prom|
+-----------------------------+-------------------+-------------------+---+---------+
|esp32-fisico-equipo01-control|2026-10-07 10:19:40|2026-10-07 10:20:00|1  |15.7     |
+-----------------------------+-------------------+-------------------+---+---------+

== Pero SÍ se publicaron los dos en el topic ==
+------+-----------------------------+--------------+-----------+-------+-------------------+
|offset|sensorId                     |origen        |temperatura|humedad|ts                 |
+------+-----------------------------+--------------+-----------+-------+-------------------+
|625   |esp32-fisico-equipo01-control|prueba-control|15.7       |36.0   |2026-10-07 10:19:40|
|626   |esp32-fisico-equipo01-tardio |prueba-tardio |15.7       |36.0   |2026-10-0

**Captura sugerida:** las dos tablas juntas: el evento tardío publicado en el topic (con su `ts` viejo) y su ausencia en la salida agregada, con el control presente.

## Tarea 4 - Un evento duplicado que `dropDuplicates` filtra
**Producto:** el mismo evento (valores reales, mismo `sensorId` y mismo `timestamp`) publicado **dos veces**, y contado **una sola vez** por una consulta con `dropDuplicates` acotado por watermark. Arranca la consulta **antes** de publicar.

In [11]:
deduplicado = (
    eventos
    .withWatermark("ts", "30 seconds")
    .dropDuplicates(["sensorId", "timestamp"])
)
consulta_dedup = (
    deduplicado.writeStream
    .outputMode("append")
    .format("memory").queryName("dedup_clima")
    .trigger(processingTime="3 seconds")
    .start()
)
print("consulta_dedup activa:", consulta_dedup.isActive)
time.sleep(10)   # deja que arranque antes de publicar

consulta_dedup activa: True


In [12]:
real = ultimo_evento_real()
for _ in range(2):   # mismo evento, dos veces
    publicar_evento(SENSOR_ID + "-duplicado", real["temperatura"], real["humedad"], real["timestamp"], "prueba-duplicado")
print(">>> '-duplicado' publicado DOS veces (mismo sensorId y mismo timestamp)")

>>> '-duplicado' publicado DOS veces (mismo sensorId y mismo timestamp)


In [18]:
time.sleep(15)
marca = SENSOR_ID + "-duplicado"
en_topic = leer_topico_batch().where(col("sensorId") == marca).count()
en_salida = spark.sql(f"SELECT count(*) AS n FROM dedup_clima WHERE sensorId = '{marca}'").collect()[0]["n"]
print("veces que está en el topic:               ", en_topic)
print("veces que está en la salida deduplicada: ", en_salida)

spark.sql(f"SELECT sensorId, temperatura, humedad, timestamp FROM dedup_clima WHERE sensorId = '{marca}'").show(truncate=False)

veces que está en el topic:                2
veces que está en la salida deduplicada:  1
+-------------------------------+-----------+-------+-------------+
|sensorId                       |temperatura|humedad|timestamp    |
+-------------------------------+-----------+-------+-------------+
|esp32-fisico-equipo01-duplicado|15.7       |36.0   |1791386545000|
+-------------------------------+-----------+-------+-------------+



In [14]:
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

**Captura sugerida:** las dos líneas de conteo (`2` en el topic, `1` en la salida) y la fila única de la salida deduplicada.

## Tarea 5 - Dos intervalos de disparo sobre mi propio stream
**Producto:** el mismo stream con trigger de 1 s y de 6 s, ambos durante **exactamente 15 s de reloj** (`awaitTermination`), y el `batchId` alcanzado por cada uno.

> Esta celda **detiene las consultas anteriores** (para que no compitan por CPU y la comparación sea justa). Asegúrate de haber capturado ya la evidencia de las tareas 2, 3 y 4.

In [20]:
detener_todas()


def contar_microlotes(intervalo, segundos):
    q = (
        eventos.writeStream
        .outputMode("append")
        .format("memory")
        .queryName("conteo_" + intervalo.replace(" ", "_"))
        .trigger(processingTime=intervalo)
        .start()
    )
    q.awaitTermination(segundos)
    q.stop()
    lp = q.lastProgress
    return {
        "intervalo": intervalo,
        "ultimo_batchId": lp["batchId"] if lp else None,
        "filas_procesadas": sum(p["numInputRows"] for p in q.recentProgress),
    }


resultados = [contar_microlotes("1 second", 15), contar_microlotes("6 seconds", 15)]
for r in resultados:
    print(r)

deteniendo consulta: agg_clima
deteniendo consulta: dedup_clima
{'intervalo': '1 second', 'ultimo_batchId': 4, 'filas_procesadas': 4}
{'intervalo': '6 seconds', 'ultimo_batchId': 2, 'filas_procesadas': 3}


**Cómo interpretarlo:** el ESP32 publica una lectura cada 3 s, así que un trigger de 1 s **no puede** generar un micro-lote por segundo: solo hay datos nuevos cada 3 s. Con 6 s cada lote agrupa ~2 lecturas y hay menos lotes, es decir menos overhead pero más espera por evento. Anota esa observación en tu informe: es un hallazgo real de tu fuente, no de un simulador.

**Captura sugerida:** la celda con los dos diccionarios (`ultimo_batchId` de cada intervalo).

## Tarea 6 - Persistir en Parquet con checkpoint, y verificar que acumula
**Producto:** la agregación por ventana guardada en Parquet (modo `append`, con `checkpointLocation`), ejecutada en **dos corridas sucesivas**: la segunda debe **sumar filas** (no sobreescribir) y retomar el `batchId` donde quedó la primera. Se persiste el agregado (no el crudo) porque así el checkpoint contiene también `state/`.

> No borres `./artifacts` entre las dos corridas: ahí está la salida y el checkpoint.

In [21]:
def corrida_parquet(segundos):
    q = (
        construir_agregado(eventos).writeStream
        .outputMode("append")
        .format("parquet")
        .option("path", RUTA_PARQUET)
        .option("checkpointLocation", CHK_PARQUET)
        .trigger(processingTime="5 seconds")
        .start()
    )
    q.awaitTermination(segundos)
    q.stop()
    lp = q.lastProgress
    return lp["batchId"] if lp else None


def resumen_parquet():
    try:
        df = spark.read.parquet(RUTA_PARQUET)
    except Exception:
        print("Aún no hay Parquet escrito")
        return 0
    n = df.count()
    print("filas guardadas en Parquet:", n)
    (
        df.select(col("window.start").alias("inicio"), col("window.end").alias("fin"), "sensorId", "n",
                  F.round("temp_prom", 2).alias("temp_prom"))
        .orderBy("inicio").show(50, truncate=False)
    )
    return n


def inspeccionar_checkpoint(chk=CHK_PARQUET):
    print("carpetas del checkpoint:", sorted(f for f in os.listdir(chk) if not f.startswith(".")))
    d = os.path.join(chk, "offsets")
    lotes = sorted((f for f in os.listdir(d) if f.isdigit()), key=int)   # orden numérico, ignora los .crc
    with open(os.path.join(d, lotes[-1])) as f:
        lineas = f.read().strip().split("\n")
    meta = json.loads(lineas[1])
    print("\n[offsets/] último micro-lote registrado:", lotes[-1])
    print("  batchWatermarkMs:", meta.get("batchWatermarkMs"), "| batchTimestampMs:", meta.get("batchTimestampMs"))
    print("  offsets de Kafka leídos:", lineas[2:])
    commits = sorted((f for f in os.listdir(os.path.join(chk, "commits")) if f.isdigit()), key=int)
    print("\n[commits/] últimos lotes confirmados:", commits[-5:])
    print("[state/] operadores con estado:", sorted(f for f in os.listdir(os.path.join(chk, "state")) if not f.startswith(".")))
    return int(lotes[-1])

### Corrida 1 (~90 s)

In [22]:
batch_corrida1 = corrida_parquet(90)
filas_corrida1 = resumen_parquet()
print("\nCORRIDA 1 -> último batchId:", batch_corrida1, "| filas en Parquet:", filas_corrida1)

26/10/07 16:10:00 ERROR FileFormatWriter: Aborting job 4117e5e5-e4c9-4892-8900-a716aa09e572.
java.lang.InterruptedException
	at java.base/java.util.concurrent.locks.AbstractQueuedSynchronizer.tryAcquireSharedNanos(AbstractQueuedSynchronizer.java:1167)
	at scala.concurrent.impl.Promise$DefaultPromise.tryAwait0(Promise.scala:251)
	at scala.concurrent.impl.Promise$DefaultPromise.result(Promise.scala:271)
	at org.apache.spark.util.SparkThreadUtils$.awaitResultNoSparkExceptionConversion(SparkThreadUtils.scala:78)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.B

filas guardadas en Parquet: 3
+-------------------+-------------------+---------------------+---+---------+
|inicio             |fin                |sensorId             |n  |temp_prom|
+-------------------+-------------------+---------------------+---+---------+
|2026-10-07 11:08:20|2026-10-07 11:08:40|esp32-fisico-equipo01|3  |16.1     |
|2026-10-07 11:08:40|2026-10-07 11:09:00|esp32-fisico-equipo01|7  |16.1     |
|2026-10-07 11:09:00|2026-10-07 11:09:20|esp32-fisico-equipo01|6  |16.1     |
+-------------------+-------------------+---------------------+---+---------+


CORRIDA 1 -> último batchId: 17 | filas en Parquet: 3


### Contenido real del checkpoint (offsets/, commits/, state/)

In [23]:
ultimo_lote_chk = inspeccionar_checkpoint()

carpetas del checkpoint: ['commits', 'metadata', 'offsets', 'sources', 'state']

[offsets/] último micro-lote registrado: 18
  batchWatermarkMs: 1791389383000 | batchTimestampMs: 1791389400004
  offsets de Kafka leídos: ['{"clima-real-eventos":{"0":1467}}']

[commits/] últimos lotes confirmados: ['13', '14', '15', '16', '17']
[state/] operadores con estado: ['0']


**Captura sugerida (checkpoint):** la salida de la celda anterior. `offsets/` guarda hasta qué offset de Kafka se leyó, `commits/` qué lotes terminaron, `state/` el contenido de las ventanas todavía abiertas.

### Corrida 2
Deja pasar **1 o 2 minutos** con el ESP32 publicando (así hay datos nuevos acumulados en el topic mientras no había consulta) y ejecuta la siguiente celda. Al arrancar apuntando al **mismo checkpoint**, la consulta debe retomar desde el último offset confirmado.

In [24]:
batch_corrida2 = corrida_parquet(90)
filas_corrida2 = resumen_parquet()
print("\nCORRIDA 1 -> filas:", filas_corrida1, "| último batchId:", batch_corrida1)
print("CORRIDA 2 -> filas:", filas_corrida2, "| último batchId:", batch_corrida2)
print("¿acumula (no sobreescribe)?:", filas_corrida2 > filas_corrida1)
print("¿retomó el batchId desde el checkpoint (no volvió a 0)?:", batch_corrida2 is not None and batch_corrida2 > ultimo_lote_chk)

filas guardadas en Parquet: 15
+-------------------+-------------------+---------------------+---+---------+
|inicio             |fin                |sensorId             |n  |temp_prom|
+-------------------+-------------------+---------------------+---+---------+
|2026-10-07 11:08:20|2026-10-07 11:08:40|esp32-fisico-equipo01|3  |16.1     |
|2026-10-07 11:08:40|2026-10-07 11:09:00|esp32-fisico-equipo01|7  |16.1     |
|2026-10-07 11:09:00|2026-10-07 11:09:20|esp32-fisico-equipo01|6  |16.1     |
|2026-10-07 11:09:20|2026-10-07 11:09:40|esp32-fisico-equipo01|7  |16.1     |
|2026-10-07 11:09:40|2026-10-07 11:10:00|esp32-fisico-equipo01|7  |16.1     |
|2026-10-07 11:10:00|2026-10-07 11:10:20|esp32-fisico-equipo01|6  |16.1     |
|2026-10-07 11:10:20|2026-10-07 11:10:40|esp32-fisico-equipo01|7  |16.1     |
|2026-10-07 11:10:40|2026-10-07 11:11:00|esp32-fisico-equipo01|7  |16.1     |
|2026-10-07 11:11:00|2026-10-07 11:11:20|esp32-fisico-equipo01|6  |16.1     |
|2026-10-07 11:11:20|2026-10-07 1

**Captura sugerida:** las cuatro líneas finales (filas y `batchId` de cada corrida, y los dos `True`).

## Hallazgo: una consulta con estado que no arranca sin `checkpointLocation`
Evidencia lista para el apartado "Error o hallazgo" del informe: el mismo `writeStream` a Parquet, pero **sin** `checkpointLocation`.

In [26]:
try:
    (
        construir_agregado(eventos).writeStream
        .outputMode("append")
        .format("parquet")
        .option("path", "./artifacts/sin_checkpoint")
        .start()
    )
    print("La consulta arrancó (inesperado)")
except Exception as e:
    print(type(e).__name__)
    print(str(e).split("\n")[0])

AnalysisException
checkpointLocation must be specified either through option("checkpointLocation", ...) or SparkSession.conf.set("spark.sql.streaming.checkpointLocation", ...).


Si en tu corrida te pasó otra cosa distinta (un evento tardío que **sí** pasó por arranque en frío, una corrida con cero filas porque el bridge o el ESP32 estaban detenidos, la contraseña de Jupyter, etc.), **ese** es un hallazgo más valioso: descríbelo con lo que realmente viste.

## Cierre

In [27]:
detener_todas()
print("consultas activas:", len(spark.streams.active))

consultas activas: 0


### Qué evidencia va en cada bloque de la rúbrica (4.6)

| Bloque de la rúbrica | Celdas con la evidencia |
|---|---|
| 1. Lectura streaming y agregación por ventana | Tarea 1 (esquema y datos reales) + Tarea 2 (ventanas con watermark) + celda de latencia (justifica el margen) |
| 2. Dato tardío y deduplicación | Tarea 3 (control presente, tardío ausente) + Tarea 4 (2 en el topic, 1 en la salida) |
| 3. Intervalo de disparo y checkpoint | Tarea 5 (`batchId` por intervalo) + inspección del checkpoint (Tarea 6) |
| 4. Persistencia en Parquet | Tarea 6, corridas 1 y 2 (filas que acumulan, `batchId` que continúa) |
| Error o hallazgo | Celda de hallazgo, u otro problema real que hayas tenido |

### Antes de capturar pantalla
- Cada captura **sin recortar**, con el **reloj de Windows** y tu **usuario o foto de perfil** visibles.
- Las horas deben ser **coherentes con tus commits**: sube a GitHub este notebook y el sketch `esp32_dht.ino` el mismo día en que corres las pruebas.

### Para la reflexión (5 a 8 líneas)
Usa tus propios números: el **p95 de latencia** medido, el margen que elegiste, qué ventana cerró más tarde por ese margen, y qué dato tardío o duplicado dejaste de ver con esa decisión.

In [ ]:
# OPCIONAL - solo si quieres repetir la Tarea 6 desde cero (borra el Parquet y su checkpoint). Descomenta para usar.
# detener_todas()
# reiniciar_checkpoint(RUTA_PARQUET, CHK_PARQUET)